# Extraction

In [ ]:
import torch
import gc
from extraction_tools import *
from models import *
from glob import glob

save_dir = "path/to/dataset"
vid_paths = glob("path/to/dataset/vids")

keyframe_extractor = KeyframeAndFeatureExtractor(
    SigLIP2Model(name="google/siglip2-giant-opt-patch16-384", device="cuda"),
    save_dir
)

# Customize the parameters to fit your hardware setup
keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    feature_extraction_batch_size=16,
    shot_confidence_score=0.05,
    min_keyframe_distance=0.5,
    clustering_similarity_threshold=0.98,
    num_workers=4
)

# If your VRAM is not plenty, you can delete the unused KeyframeAndFeatureExtractor after use
del keyframe_extractor
gc.collect()
torch.cuda.empty_cache()

map_paths = glob("path/to/dataset/maps")

transcript_extractor = TranscriptAndFeatureProcessor(
    FasterWhisperModel(name="turbo", device="cuda"),
    BGEM3Model(name="BAAI/bge-m3", device="cuda"),
    save_dir
)

# Customize the parameters to fit your hardware setup
transcript_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    max_sentence_duration=12,
    alignment_transcript_time_range=24,
    feature_extraction_batch_size=32,
    num_workers=4
)

# If your VRAM is not plenty, you can delete the unused TranscriptAndFeatureProcessor after use
del transcript_extractor
gc.collect()
torch.cuda.empty_cache()

# Uploading

In [ ]:
from db_management_tools import DataLoader

dataset_dir = "path/to/dataset"
data_loader = DataLoader(dataset_dir)
data_loader.create_and_fill_database(
    "collection_name",
    upload_batch_size=1024,
    upload_num_workers=8
)

# Query testing

In [ ]:
from db_management_tools import QueryManager
from models import SigLIP2Model, BGEM3Model

# For result display
from peripheral_tools import NotebookDisplay 

query_manager = QueryManager(
	SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda"),
    BGEM3Model(name="BAAI/bge-m3", device="cuda")
)

query = input("Enter your query: ")
limit = int(input("Enter your search limit: "))
points = query_manager.search(
    "collection_name",
    query=query,
    search_limit=limit,
)

display = NotebookDisplay(315, 5)
display.display(points)